# TryLM: Training a Small Language Model

This notebook trains a ~10M parameter GPT-style language model on the TinyStories dataset.

**Features:**
- Custom BPE tokenizer
- GPT architecture with RoPE and SwiGLU
- Mixed precision training
- W&B logging

**Requirements:**
- GPU runtime (T4/P100/V100)
- ~4GB VRAM

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repository (if running from Colab)
import os

if not os.path.exists('TryLM'):
    !git clone https://github.com/YOUR_USERNAME/TryLM.git
    %cd TryLM
else:
    %cd TryLM

In [ ]:
# Install dependencies
!pip install -q torch wandb tokenizers datasets safetensors tqdm pyyaml rich typer

In [ ]:
# Install the package in development mode
!pip install -e .

In [ ]:
# Login to W&B
import wandb
wandb.login()

## 2. Train Tokenizer

In [ ]:
from trylm.config import TokenizerConfig
from trylm.tokenizer import train_tokenizer_on_dataset

# Configure tokenizer
tokenizer_config = TokenizerConfig(
    vocab_size=16384,
    min_frequency=2,
)

# Train tokenizer
tokenizer = train_tokenizer_on_dataset(
    dataset_name="roneneldan/TinyStories",
    config=tokenizer_config,
)

# Save tokenizer
tokenizer.save("data/tokenizer")

print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Test tokenizer
test_text = "Once upon a time, there was a little girl."
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens)

print(f"Original: {test_text}")
print(f"Tokens: {tokens}")
print(f"Decoded: {decoded}")
print(f"Num tokens: {len(tokens)}")

## 3. Configure Model and Training

In [ ]:
from trylm.config import Config, ModelConfig, TrainConfig

# Configure model (~10M params)
model_config = ModelConfig(
    vocab_size=tokenizer.vocab_size,
    n_layers=6,
    n_heads=6,
    d_model=384,
    context_length=512,
    dropout=0.1,
)

# Configure training
train_config = TrainConfig(
    batch_size=64,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    max_steps=10000,
    warmup_steps=100,
    eval_interval=500,
    save_interval=1000,
    mixed_precision=True,
    compile=True,
    wandb_project="trylm-colab",
)

config = Config(
    tokenizer=tokenizer_config,
    model=model_config,
    train=train_config,
)

# Print parameter count estimate
params = model_config.count_parameters()
print(f"Estimated parameters: {params['total']:,}")

## 4. Create Model and Dataloaders

In [ ]:
import torch
from trylm.model import GPT
from trylm.data import create_dataloaders

# Create model
model = GPT.from_config(model_config)
print(f"Actual parameters: {model.count_parameters():,}")

# Create dataloaders
train_loader, val_loader = create_dataloaders(
    tokenizer=tokenizer,
    model_config=model_config,
    train_config=train_config,
    streaming=False,  # Set to True for very large datasets
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 5. Train!

In [ ]:
from trylm.trainer import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
)

# Start training
trainer.train()

## 6. Generate Text

In [ ]:
from trylm.generate import generate

# Load best model
from trylm.evaluate import load_model_for_eval
from pathlib import Path

model, tokenizer, config = load_model_for_eval(Path("checkpoints/best"), device)

In [ ]:
# Generate some stories!
prompts = [
    "Once upon a time",
    "The little girl loved to",
    "One sunny day, a boy named Tom",
    "In a magical forest",
]

for prompt in prompts:
    text = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=100,
        temperature=0.8,
        top_p=0.9,
        device=device,
    )
    print(f"\n{'='*50}")
    print(text)
    print(f"{'='*50}")

## 7. Evaluate

In [ ]:
from trylm.evaluate import evaluate_model

metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    config=config,
    device=device,
    split="validation",
    max_batches=100,
)

print(f"\nValidation Loss: {metrics['loss']:.4f}")
print(f"Validation Perplexity: {metrics['perplexity']:.2f}")

## 8. Download Model (Optional)

Download the trained model to use locally.

In [ ]:
# Zip checkpoints for download
!zip -r trylm_model.zip checkpoints/best data/tokenizer

# For Colab: download the zip file
try:
    from google.colab import files
    files.download('trylm_model.zip')
except:
    print("Not running in Colab. Find the zip file in the current directory.")